In [1]:
import os

# === 国内网络环境配置 ===
# Unsloth 默认从 HuggingFace Hub 下载统计数据和模型，在国内会超时卡死。
# 以下环境变量必须在 import unsloth/transformers 之前设置。
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"   # Unsloth 使用 ModelScope，跳过 HF 统计
os.environ["HF_HUB_OFFLINE"] = "1"           # 禁止 huggingface_hub 发起任何网络请求
os.environ["TRANSFORMERS_OFFLINE"] = "1"      # 禁止 transformers 联网下载
os.environ["HF_DATASETS_OFFLINE"] = "1"       # 禁止 datasets 联网下载

from pathlib import Path
import sys

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        project_root = candidate
        break

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

os.chdir(project_root)

### 加载
#### 加载模型

In [2]:
from unsloth import FastModel
import torch
from gmsp.clients import create_transport_client
from gmsp.config import load_gmsp_config, get_default_profile_name, get_profile
from gmsp.tracking.experiment_tracking import create_experiment_tracker, ExperimentTrackingCallback
from gmsp.training.rl_training import (
    build_reward_functions,
    build_training_args,
    describe_training_recipe,
    get_policy_trainer_class,
)
import uuid

gmsp_config = load_gmsp_config()
default_profile_name = get_default_profile_name(gmsp_config)
profile = get_profile(gmsp_config, default_profile_name)
transport_config = profile["transport"]
model_config = profile["model"]
lora_config = profile["lora"]
training_config = profile["training"]
run_tracker = create_experiment_tracker(default_profile_name, profile, gmsp_config)
tracking_callback = ExperimentTrackingCallback(run_tracker)

# ?????????
client = create_transport_client(transport_config)

max_seq_length = model_config["max_seq_length"]
lora_rank = model_config["lora_rank"]

model, tokenizer = FastModel.from_pretrained(
    model_name = model_config["model_name"],
    max_seq_length = max_seq_length,
    load_in_4bit = model_config["load_in_4bit"],
    load_in_8bit = model_config["load_in_8bit"],
    full_finetuning = model_config.get("full_finetuning", False),
)



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/root/miniconda3/lib/python3.12/site-packages/unsloth_zoo/temporary_patches/__init__.py:25: UserWarning: Unsloth Zoo: skipping gemma3n temporary patches due to import error: duplicate template name
  warnings.warn(f"Unsloth Zoo: skipping gemma3n temporary patches due to import error: {e}")


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


==((====))==  Unsloth 2026.3.18: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 47.371 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

#### 加载 Lora 设置

In [3]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # 仅处理文本层或者模型没有视觉层时关闭
    finetune_language_layers   = True,  # 应该保持开启！
    finetune_attention_modules = True,  # 注意力机制对DAPO有好处
    finetune_mlp_modules       = True,  # 应该始终保持开启！

    r = lora_rank,           # 更大 = 更高的精度，但可能过拟合
    lora_alpha = lora_rank,  # 建议alpha至少等于r
    lora_dropout = lora_config["lora_dropout"],
    use_gradient_checkpointing = model_config.get("use_gradient_checkpointing", "unsloth"),
    bias = lora_config["bias"],
    random_state = lora_config["random_state"], # 使用同一个随机数种子
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


#### 加载、构造数据集

##### 构造系统提示词

In [4]:
# 设置系统提示词
reasoning_start = "<think>"
reasoning_end   = "</think>"
solution_start = "<code>"
solution_end   = "</code>"

system_prompt = \
f"""你是一个 Blender 材质生成器。你的任务是根据用户描述，用 bpy Python 代码创建对应的 Blender 节点材质。

执行环境说明：
- 代码运行时，变量 `material` 已经是一个启用了 use_nodes 的空材质对象（节点树已清空）。
- 你可以直接使用 `material.node_tree.nodes` 和 `material.node_tree.links` 来构建节点。
- 不需要调用 bpy.data.materials.new()，也不需要将材质赋给任何对象。
- 只能使用 bpy 和 mathutils 模块，不能使用外部文件或其他库。

请将思考过程放在 {reasoning_start} 和 {reasoning_end} 之间。
然后，请在 {solution_start} 和 {solution_end} 之间提供你的答案。"""
system_prompt

'你是一个 Blender 材质生成器。你的任务是根据用户描述，用 bpy Python 代码创建对应的 Blender 节点材质。\n\n执行环境说明：\n- 代码运行时，变量 `material` 已经是一个启用了 use_nodes 的空材质对象（节点树已清空）。\n- 你可以直接使用 `material.node_tree.nodes` 和 `material.node_tree.links` 来构建节点。\n- 不需要调用 bpy.data.materials.new()，也不需要将材质赋给任何对象。\n- 只能使用 bpy 和 mathutils 模块，不能使用外部文件或其他库。\n\n请将思考过程放在 <think> 和 </think> 之间。\n然后，请在 <code> 和 </code> 之间提供你的答案。'

##### 构造数据集

In [5]:
from datasets import Dataset
import random

dataset = []

# 单一训练目标：真实皮肤材质
task = "用bpy来生成真实的Blender皮肤材质"

# 任务前缀列表
user_start = [
    "做这个材质:", "帮我生成这个：", "帮我生成", "我希望你能帮助我生成",
    "请帮我做一个材质", "希望你帮我生成", "请你为我做一个", "你能做一下这个吗?",
    "", "请完成以下任务:", "帮我搞一个材质:", "能否帮我生成", "请协助我生成一个",
]

# 构造数据集
for user in user_start:
    dataset.append({
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user + task},
        ],
        "taskid": uuid.uuid4().hex,
        "goal": task,
    })

# 输出最终数据集
final_dataset = Dataset.from_list(dataset)
print(f"数据集大小: {len(final_dataset)}")

数据集大小: 13


### 定义奖励函数
#### 定义标准格式形式

In [6]:
import re

# 定义正则表达式，用来判断模型的输出是否符合格式要求
match_format = re.compile(
    rf"^[\s]{{0,}}"\
    rf"<think>.+?</think>.*?"\
    rf"<code>(.+?)</code>"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)

match_format.search(
    "<think>Let me think!</think>"\
    "<code>2</code>",
)

<re.Match object; span=(0, 42), match='<think>Let me think!</think><code>2</code>'>

#### 构造奖励函数

In [7]:
# 严格格式判断函数
def match_format_exactly(*, completions, **kwargs):
    """格式判断函数，严格判断格式是否匹配
    """
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

In [8]:
# 弱格式判断函数
def match_format_approximately(*, prompts, completions, **kwargs):
    """弱格式判断奖励，即使没有严格对应，也可以根据使用的标签数量来做出相应的奖励
    """
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]
        
    print('*'*20, f"Question:\n{question}", f"\nResponse:\n{responses[0]}")
    
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # 数一数看到多少个关键词——如果太多，我们会惩罚你！
        # 如果我们看到1个关键词，那么加一些积分！如果更多了，那么就应当扣除一些分
        score += 0.5 if response.count(reasoning_start) == 1 else -0.5
        score += 0.5 if response.count(reasoning_end)   == 1 else -0.5
        score += 0.5 if response.count(solution_start)  == 1 else -0.5
        score += 0.5 if response.count(solution_end)    == 1 else -0.5
        scores.append(score)
    return scores

In [9]:
# 获取代码
def code_extractor(code):
    match = re.search(r'<code>(.*?)</code>', code, flags=re.DOTALL)
    if not match:
        return ""
    content = match.group(1).strip()
    if not content:
        return ""
    return content


class BlenderResultCache:
    """缓存 Blender 结果，避免同一 step 内重复发送请求。
    
    三个远程奖励函数 (accuracy_reward, meaning_reward, error_check) 需要相同的
    Blender 渲染结果。此缓存确保每组 completions 只发送一次请求。
    """

    def __init__(self, transport_client):
        self._client = transport_client
        self._cache = {}

    def get_result(self, goal, taskid, completions):
        contents = tuple(c[0]["content"] for c in completions)
        cache_key = (taskid, contents)

        if cache_key in self._cache:
            return self._cache[cache_key]

        names = []
        materials = {
            "head": {"input": goal, "taskid": taskid, "request": []},
            "outputs": [],
        }
        for completion in completions:
            code = code_extractor(completion[0]["content"])
            name = f"M{len(materials['outputs']) + 1}"
            names.append(name)
            materials["outputs"].append({"name": name, "code": code})

        result = self._client.send_materials(materials)
        entry = {"result": result, "names": names, "materials": materials}
        self._cache[cache_key] = entry
        return entry

    def clear(self):
        self._cache.clear()


blender_cache = BlenderResultCache(client)

In [10]:
# 生成的图像和描述的相似度
def accuracy_reward(*, completions, goal, taskid, **kwargs):
    """计算生成的图像和描述的相似度
    
    返回原始分数，不做归一化 — TRL GRPOTrainer 会自动做 group-wise 归一化。
    """
    WEIGHT = 2

    entry = blender_cache.get_result(goal[0], taskid[0], completions)
    c = entry["result"]
    names = entry["names"]
    materials = entry["materials"]

    # send_materials 失败时返回 list，成功时返回 dict
    if isinstance(c, list):
        print("[WARN] accuracy_reward: send_materials 返回了错误列表，全部给 0 分")
        scores = [0.0] * len(names)
        run_tracker.log_reward_event(
            "accuracy_reward", materials, c, scores,
            extra={"goal": goal[0], "taskid": taskid[0], "names": names, "error": "send_materials returned list"},
        )
        return scores

    results = c.get("accuracy_rank", {})
    scores = []
    max_rank = len(names)
    for name in names:
        rank = int(results.get(name, 0))
        # rank 越小越好，转换成效用分数：rank=1 得 max_rank 分，rank=0（失败）得 0 分
        scores.append((max_rank - rank + 1) * WEIGHT if rank > 0 else 0.0)

    run_tracker.log_reward_event(
        "accuracy_reward", materials, c, scores,
        extra={"goal": goal[0], "taskid": taskid[0], "names": names},
    )
    print("accuracy_reward" + str(results))
    return scores

In [11]:
# 图像是否有意义
def meaning_reward(*, completions, goal, taskid, **kwargs):
    """计算生成的图像是否有意义
    
    返回原始分数，不做归一化 — TRL GRPOTrainer 会自动做 group-wise 归一化。
    """
    WEIGHT = 1

    entry = blender_cache.get_result(goal[0], taskid[0], completions)
    c = entry["result"]
    names = entry["names"]
    materials = entry["materials"]

    # send_materials 失败时返回 list，成功时返回 dict
    if isinstance(c, list):
        print("[WARN] meaning_reward: send_materials 返回了错误列表，全部给 0 分")
        scores = [0.0] * len(names)
        run_tracker.log_reward_event(
            "meaning_reward", materials, c, scores,
            extra={"goal": goal[0], "taskid": taskid[0], "names": names, "error": "send_materials returned list"},
        )
        return scores

    results = c.get("meaning_rank", {})
    scores = []
    max_rank = len(names)
    for name in names:
        rank = int(results.get(name, 0))
        scores.append((max_rank - rank + 1) * WEIGHT if rank > 0 else 0.0)

    run_tracker.log_reward_event(
        "meaning_reward", materials, c, scores,
        extra={"goal": goal[0], "taskid": taskid[0], "names": names},
    )
    print("meaning_reward" + str(results))
    return scores

In [12]:
# 代码是否报错
def error_check(*, completions, goal, taskid, **kwargs):
    """检查生成的代码是否报错
    
    成功 = WEIGHT 分，失败 = 0 分。不做归一化。
    当所有 completion 都成功时，它们都拿到 WEIGHT — TRL 的 group-wise 归一化
    会让 advantage=0，这是正确的：error_check 不需要区分，其他奖励函数会区分质量。
    """
    WEIGHT = 2

    entry = blender_cache.get_result(goal[0], taskid[0], completions)
    c = entry["result"]
    names = entry["names"]
    materials = entry["materials"]

    # send_materials 失败时返回 list，成功时返回 dict
    if isinstance(c, list):
        print("[WARN] error_check: send_materials 返回了错误列表，全部给 0 分")
        scores = [0.0] * len(names)
        run_tracker.log_reward_event(
            "error_check", materials, c, scores,
            extra={"goal": goal[0], "taskid": taskid[0], "names": names, "error": "send_materials returned list"},
        )
        return scores

    results = c.get("status", {})
    scores = []
    for name in names:
        success = results.get(name, False)
        scores.append(WEIGHT if success else 0.0)

    run_tracker.log_reward_event(
        "error_check", materials, c, scores,
        extra={"goal": goal[0], "taskid": taskid[0], "names": names},
    )
    print("error_check" + str(results))
    return scores

### 训练部分
#### 训练配置

In [13]:
max_prompt_length = training_config["max_prompt_length"]

# ?? DAPO ????????
training_args = build_training_args(
    training_config,
    max_prompt_length = max_prompt_length,
    max_seq_length = max_seq_length,
    output_dir = str(run_tracker.output_dir),
)
max_completion_length = getattr(
    training_args,
    "max_completion_length",
    max_seq_length - max_prompt_length,
)


#### 开始训练

In [14]:
# ??????????????? reward function
run_tracker.log_dataset_summary(
    final_dataset,
    split="train",
    extra={
        "max_seq_length": max_seq_length,
        "max_prompt_length": max_prompt_length,
        "algorithm": training_config.get("algorithm", "dapo"),
    },
)
base_reward_funcs = [
    # ????
    match_format_exactly,
    match_format_approximately,

    # ??????
    accuracy_reward,
    meaning_reward,

    # ??????
    error_check,
]
reward_funcs = build_reward_functions(
    base_reward_funcs,
    training_config,
    max_completion_length = max_completion_length,
)
run_tracker.write_json(
    "rl_recipe.json",
    describe_training_recipe(
        training_config,
        max_prompt_length = max_prompt_length,
        max_completion_length = max_completion_length,
        reward_functions = reward_funcs,
    ),
)
trainer_cls = get_policy_trainer_class()
trainer = trainer_cls(
    model = model,
    processing_class = tokenizer,
    reward_funcs = reward_funcs,
    callbacks = [tracking_callback],
    args = training_args,
    train_dataset = final_dataset,
)

# Unsloth 会往 model.config 注入 function 对象，导致 TensorBoard 序列化失败
# 在训练前清理掉不可 JSON 序列化的字段
import json as _json
def _sanitize_config(config):
    d = config.to_dict()
    for k, v in list(d.items()):
        if callable(v) and not isinstance(v, type):
            setattr(config, k, None)
        elif isinstance(v, dict):
            for k2, v2 in list(v.items()):
                if callable(v2) and not isinstance(v2, type):
                    v[k2] = None
            setattr(config, k, v)

_sanitize_config(trainer.model.config)
if hasattr(trainer.model, "base_model"):
    _base = trainer.model.base_model
    if hasattr(_base, "model") and hasattr(_base.model, "config"):
        _sanitize_config(_base.model.config)

try:
    trainer.train()
except Exception as exc:
    run_tracker.finalize("failed", {"error": str(exc)})
    raise

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13 | Num Epochs = 93 | Total steps = 1,200
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 1 x 1) = 6
 "-____-"     Trainable parameters = 86,556,672 of 9,496,370,416 (0.91% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


******************** Question:
希望你帮我生成用bpy来生成真实的Blender皮肤材质 
Response:
用户希望我生成一个使用 bpy 创建真实的 Blender 人体皮肤材质的代码。

真实的皮肤材质需要考虑多个物理特性：

1. **表面纹理** - 皮肤不是完美的光滑表面，应该有轻微的纹理细节
2. **次表面散射 (SSS)** - 这是皮肤最重要的特性，光线可以穿透皮肤表面散射，产生那种"透光"的效果
3. **多层结构** - 皮肤有表皮层、真皮层等，可以用多层混合实现
4. **颜色变化** - 皮肤颜色可以有微妙的变化，包括红色/血色、黄色/肤色、棕色等
5. **法线/凹凸贴图** - 皮肤表面不是完全平滑的
6. **粗糙度变化** - 不同区域的皮肤粗糙度不同（眼角、嘴唇等）
7. **高光** - 皮肤的高光应该比较柔和

我将创建一个完整的皮肤材质节点树，包括：
- 主颜色节点（混合多个颜色层）
- 次表面散射设置
- 多层结构模拟
- 法线/凹凸节点
- 粗糙度控制
- 适当的 AO/环境光遮蔽

让我编写完整的代码。
</think>

<code>
import bpy
import mathutils

# 创建皮肤材质
def create_realistic_skin_material():
    
    # 1. 创建主颜色输入节点
    input_node = material.node_tree.nodes.new('ShaderNodeBsdfPrincipled')
    material.node_tree.nodes['Principled BSDF'].default_value.location = (0, -2)
    
    # 2. 创建主肤色
    base_color = material.node_tree.nodes.new('ShaderNodeRGB')
    base_color.location = (-400, 0)
    base_color.inputs['Color'].default_value = (1.0, 0.95, 0.90)  # 基础肤色
    
    # 3. 添加皮下血色层（红色）
    blood_

OutOfMemoryError: CUDA out of memory. Tried to allocate 94.00 MiB. GPU 0 has a total capacity of 47.37 GiB of which 33.88 MiB is free. Including non-PyTorch memory, this process has 47.33 GiB memory in use. Of the allocated memory 46.77 GiB is allocated by PyTorch, and 68.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
import os
from collections import defaultdict
import seaborn as sns

# 设置Seaborn样式以获得更好看的图表
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

def extract_rewards_from_log(log_path):
    """从训练日志文件中提取奖励数据
    
    参数:
        log_path: 日志文件路径
        
    返回:
        包含步骤和对应奖励的pandas DataFrame
    """
    # 存储数据的字典
    data = defaultdict(list)
    step_pattern = re.compile(r'Step\s+(\d+)')
    reward_pattern = re.compile(r'Reward_(\d+):\s+([-\d.]+)')
    mean_reward_pattern = re.compile(r'Mean Reward:\s+([-\d.]+)')
    
    if not os.path.exists(log_path):
        print(f"日志文件 {log_path} 不存在!")
        return pd.DataFrame()
    
    with open(log_path, 'r') as f:
        for line in f:
            # 提取步骤
            step_match = step_pattern.search(line)
            if step_match:
                current_step = int(step_match.group(1))
                data['step'].append(current_step)
                
                # 提取各个奖励函数的值
                rewards = reward_pattern.findall(line)
                for idx, value in rewards:
                    data[f'reward_{idx}'].append(float(value))
                
                # 提取平均奖励
                mean_match = mean_reward_pattern.search(line)
                if mean_match:
                    data['mean_reward'].append(float(mean_match.group(1)))
    
    return pd.DataFrame(data)

def extract_rewards_from_trainer(trainer):
    """从trainer对象中直接提取奖励数据
    
    参数:
        trainer: ?????对象
        
    返回:
        包含步骤和对应奖励的pandas DataFrame
    """
    if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
        data = defaultdict(list)
        for entry in trainer.state.log_history:
            if 'step' in entry:
                data['step'].append(entry['step'])
                
                # 提取各个奖励
                for key, value in entry.items():
                    if key.startswith('reward_'):
                        data[key].append(value)
                
                # 提取平均奖励
                if 'mean_reward' in entry:
                    data['mean_reward'].append(entry['mean_reward'])
                
        return pd.DataFrame(data)
    else:
        print("训练器没有日志历史或者结构不符合预期!")
        return pd.DataFrame()

def plot_rewards(data, title="DAPO训练奖励曲线", save_path=None, moving_avg_window=5):
    """绘制奖励折线图
    
    参数:
        data: 包含奖励数据的DataFrame
        title: 图表标题
        save_path: 保存图表的路径，如果为None则显示图表
        moving_avg_window: 移动平均窗口大小
    """
    if data.empty:
        print("没有数据可以绘图!")
        return
    
    fig, ax = plt.subplots()
    
    # 定义一组专业的颜色
    colors = sns.color_palette('viridis', n_colors=len(data.columns)-1)
    
    # 绘制每个奖励函数的曲线
    for i, col in enumerate([col for col in data.columns if col != 'step']):
        # 原始数据点（透明度降低）
        ax.plot(data['step'], data[col], alpha=0.3, color=colors[i], label=f"{col} (raw)")
        
        # 添加移动平均线
        if len(data) >= moving_avg_window:
            moving_avg = data[col].rolling(window=moving_avg_window).mean()
            ax.plot(data['step'], moving_avg, linewidth=2, color=colors[i], label=f"{col} ({moving_avg_window}-point avg)")
    
    # 添加标题和标签
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel('Training Steps', fontsize=14)
    ax.set_ylabel('Reward', fontsize=14)
    
    # 添加网格线和图例
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend(loc='best', fontsize=12)
    
    # 添加统计信息
    if 'mean_reward' in data.columns:
        final_mean = data['mean_reward'].iloc[-1]
        max_mean = data['mean_reward'].max()
        min_mean = data['mean_reward'].min()
        stats_text = f"Final mean reward: {final_mean:.4f}\nMax mean reward: {max_mean:.4f}\nMin mean reward: {min_mean:.4f}"
        plt.figtext(0.02, 0.02, stats_text, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
    # 保存或显示图表
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"图表已保存到 {save_path}")
    else:
        plt.show()

# 示例用法
def visualize_rewards(trainer=None, log_file=None, output_path=None):
    """可视化训练奖励
    
    参数:
        trainer: ?????对象，如果提供则直接从训练器中提取数据
        log_file: 日志文件路径，如果trainer不可用则从日志文件中提取数据
        output_path: 图表保存路径，默认为当前目录下的'reward_plot.png'
    """
    if output_path is None:
        output_path = 'reward_plot.png'
    
    if trainer is not None:
        data = extract_rewards_from_trainer(trainer)
    elif log_file is not None:
        data = extract_rewards_from_log(log_file)
    else:
        print("请提供trainer对象或日志文件路径!")
        return
    
    plot_rewards(data, save_path=output_path)
    
    # 输出一些统计信息
    if not data.empty and 'mean_reward' in data.columns:
        print("\n--- 奖励统计信息 ---")
        print(f"最终平均奖励: {data['mean_reward'].iloc[-1]:.4f}")
        print(f"最大平均奖励: {data['mean_reward'].max():.4f}")
        print(f"最小平均奖励: {data['mean_reward'].min():.4f}")
        
        # 计算奖励增长率
        if len(data) > 1:
            first_reward = data['mean_reward'].iloc[0]
            last_reward = data['mean_reward'].iloc[-1]
            growth = ((last_reward - first_reward) / abs(first_reward)) * 100 if first_reward != 0 else float('inf')
            print(f"奖励增长率: {growth:.2f}%")

# 用法示例
# 1. 使用训练器对象
# visualize_rewards(trainer=trainer)

# 2. 或者使用日志文件
# visualize_rewards(log_file="./outputs_gemma-3_grpo_lora/opt_gemm3_2.log")

# 从训练后直接可视化
# 在训练后调用以下代码即可直接可视化
visualize_rewards(trainer=trainer, output_path="reward_trends.png")

### 模型测试
#### 默认模型测试

In [ ]:
messages = [
    # {"role": "system", "content": "你是一个 GLSL Shader 生成器，你生成出来的应当就是最终结果，可以直接使用，你也不会使用任何外部文件，纯粹程序化生成"},
    # {"role": "system", "content": "你是一个 blender 节点解释器，会和我解释 blender 节点是干什么用的"},
    {"role": "system", "content": "你是一个 Blender 的材质生成器，会直接生成材质对应的 Python 代码，该代码应该可以且仅在 Blender 中创建对应材质，你生成出来的应当就是最终结果，用户可以直接使用，不需要用户更改，你也不会使用任何外部文件"},
    {"role": "user",   "content": "给我生成一个的灰色渐变到黄色的材质"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024*2, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
# 加载原始模型（不包含微调）
from unsloth import FastModel
import torch

# 定义相同的参数
max_seq_length = 2048

# 重新加载原始模型（不应用LoRA权重）
original_model, original_tokenizer = FastModel.from_pretrained(
    model_name = "./models/qwen3.5-4b",  # 使用原始模型路径
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
)

# 测试问题
test_messages = [
    {"role": "system", "content": system_prompt},  # 使用之前定义的系统提示词
    {"role": "user", "content": "What is the sqrt of 101?"},  # 使用同样的测试问题以便比较
]

# 准备输入
test_text = original_tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt = True,
    tokenize = False,
)

# 使用TextStreamer直接查看输出
from transformers import TextStreamer
print("\n原始模型输出：")
_ = original_model.generate(
    **original_tokenizer(test_text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 2048,
    temperature = 1,  # 使用与微调模型相同的温度
    top_p = 0.95,
    top_k = 64,
    streamer = TextStreamer(original_tokenizer, skip_prompt = True),
)

#### finetuning 模型测试

In [ ]:
# 保存 Lora
model.save_lora("grpo_saved_lora")

#### 保存 Lora

In [ ]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")

In [ ]:
if True: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

### 保存为完整模型

##### 保存为 bf16 格式

In [ ]:
# Merge to 16bit
if True: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if True: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

In [ ]:
# 保存为 GGUF 格式
# if False:
#     model.save_pretrained_gguf(
#         "gemma-3-finetune",
#         quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
#     )

In [ ]:
# if False: # Change to True to upload GGUF
#     model.push_to_hub_gguf(
#         "gemma-3-finetune",
#         quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
#         repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
#         token = "hf_...",
#     )